# Evaluate a real local LangGraph workflow

Compile and execute an actual LangGraph StateGraph, evaluate route and answer contracts on synthetic cases, and demonstrate how a plausible response can hide an incorrect route.

**Audience:** Python developers building and evaluating enterprise AI applications. Familiarity with dictionaries, functions and JSON is assumed.

**Learning objectives:** compile and execute a typed LangGraph StateGraph; assess routes, answer facts, and execution paths separately; test independent invocation state and expose a deterministic router limitation.

**Prerequisites:** Understand Python functions, dictionaries, and type annotations. Install the collection environment, which includes the pinned LangGraph package. All graph nodes run locally without model credentials or a checkpointer service.

**Setup:** use patched Python 3.12 (3.12.16 or newer) and the audited environment in the [collection README](../../README.md). Run cells in order in a fresh kernel. All default data is newly invented for this tutorial. No customer records, private repositories, credentials, downloads, or paid calls are required. Optional integrations are disabled in their own cells and never enabled by an environment variable alone.

**Execution boundary:** the results below describe the local computation shown in this notebook. They do not establish hosted-model quality or production integration success.



Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../AI_DISCLOSURE.md), [license](../../LICENSE), and [attribution](../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Declare graph state and deterministic nodes

LangGraph is a required dependency for this notebook and is exercised directly. No LLM or hosted service participates. The router is intentionally inspectable so that failures can be attributed to graph logic rather than stochastic generation.


In [ ]:
import json
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict, total=False):
    question: str
    route: str
    evidence: str
    answer: str
    trace: list[str]


POLICIES = {
    "loan": "A loan lasts 14 days.",
    "refund": "A cancellation needs 48 hours notice.",
}


def route_node(state):
    text = state["question"].lower()
    matches = [
        name
        for name, words in {
            "loan": ["loan", "borrow"],
            "refund": ["refund", "cancel"],
        }.items()
        if any(word in text for word in words)
    ]
    return {"route": matches[0] if len(matches) == 1 else "clarify", "trace": ["route"]}


def retrieve_node(state):
    return {
        "evidence": POLICIES[state["route"]],
        "trace": state["trace"] + ["retrieve"],
    }


def answer_node(state):
    return {"answer": state["evidence"], "trace": state["trace"] + ["answer"]}


def clarify_node(state):
    return {
        "answer": "Please choose loan or refund.",
        "trace": state["trace"] + ["clarify"],
    }

## 2. Build, compile, and run the graph

A conditional edge sends supported topics to retrieval and all other cases to clarification. Every path terminates at END. The compile step validates the graph structure before the case matrix runs.


In [ ]:
builder = StateGraph(State)
for name, node in [
    ("route", route_node),
    ("retrieve", retrieve_node),
    ("answer", answer_node),
    ("clarify", clarify_node),
]:
    builder.add_node(name, node)
builder.add_edge(START, "route")
builder.add_conditional_edges(
    "route",
    lambda s: "clarify" if s["route"] == "clarify" else "retrieve",
    {"clarify": "clarify", "retrieve": "retrieve"},
)
builder.add_edge("retrieve", "answer")
builder.add_edge("answer", END)
builder.add_edge("clarify", END)
graph = builder.compile()
example = graph.invoke({"question": "How long may I borrow equipment?"})
assert example["route"] == "loan" and example["trace"] == [
    "route",
    "retrieve",
    "answer",
]
print(example)

## 3. Evaluate paths and facts independently

The answer key is separate from the graph implementation. Four families include direct requests, paraphrases, unsupported topics, and ambiguous combinations. Route correctness and answer correctness are separate columns.


In [ ]:
CASES = [
    ("loan duration", "loan", "14 days"),
    ("borrow equipment", "loan", "14 days"),
    ("refund notice", "refund", "48 hours"),
    ("cancel workshop", "refund", "48 hours"),
    ("parking availability", "clarify", "Please choose"),
    ("opening times", "clarify", "Please choose"),
    ("loan and refund", "clarify", "Please choose"),
    ("borrow then cancel", "clarify", "Please choose"),
]


def assess(question, expected_route, expected_fact):
    result = graph.invoke({"question": question}, config={"recursion_limit": 10})
    expected_path = (
        ["route", "clarify"] if expected_route == "clarify" else ["route", "retrieve", "answer"]
    )
    return {
        "question": question,
        "route_correct": result["route"] == expected_route,
        "answer_correct": expected_fact in result["answer"],
        "path_correct": result["trace"] == expected_path,
    }


metrics = [assess(*case) for case in CASES]
assert all(
    all(row[k] for k in ("route_correct", "answer_correct", "path_correct")) for row in metrics
)
print(json.dumps(metrics, indent=2))

## 4. Verify stateless isolation and characterize a limitation

No checkpointer is configured, so each invoke starts from its supplied state. The literal token router mishandles negation; the test below deliberately exposes that limitation instead of claiming the entire agent is correct.


In [ ]:
left = graph.invoke({"question": "loan"})
right = graph.invoke({"question": "refund"})
left["trace"].append("external-mutation")
assert "external-mutation" not in right["trace"]
negation = graph.invoke({"question": "I do not want a loan; explain a refund."})
assert negation["route"] == "clarify"
print(
    {
        "evaluated_cases": len(metrics),
        "separate_state_verified": True,
        "known_limitation": "Negation is not semantically resolved; ambiguous input takes clarification.",
    }
)

## Expected local results

Eight cases pass route, answer and path assertions using the actual compiled StateGraph. Independent invocations do not share mutable traces. The negated multi-topic request takes the documented clarification path.

**Try next:** add a checkpointer with separate authenticated thread ownership tests, then evaluate a structured-output model router against the same labels.


## Interpretation, limitations, and cleanup

This is genuine local LangGraph execution with deterministic nodes. It does not measure an LLM, remote tool authorization, a persistent checkpointer, or concurrent hosted execution. Replacing route_node with a model requires structured-output validation, an expanded held-out set, latency/cost tracking, and a safe fallback for unknown routes. If adding persistence, test session ownership and checkpoint deletion explicitly.

The default run keeps its artifacts in memory. Restart the kernel to release them. If you opt into a remote integration, review and remove its synthetic test records using that service's documented controls; this notebook does not delete remote data. Never save credential values or private provider output into notebook cells.


## References and provenance

All scenario text, values, evaluation cases and tutorial code were authored for this public example. No external dataset is bundled. Product names identify optional integrations and do not imply endorsement. API references were checked on 2026-10-01.

- [LangGraph graph API](https://docs.langchain.com/oss/python/langgraph/graph-api)
- [StateGraph compile reference](https://reference.langchain.com/python/langgraph/graph/state/StateGraph/compile)
